## Live Yelp Review Extraction and Analysis

In [1]:
import pickle
import pandas as pd
import scipy.sparse as sp
import re
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
with open("isolation_forest_yelp.pkl", "rb") as f:
    iso_model = pickle.load(f)

with open("tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

print("Model and vectorizer loaded")

Model and vectorizer loaded


In [3]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [4]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-z\s]", "", text)

    tokens = text.split()

    cleaned_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(cleaned_tokens)

In [15]:
def create_driver():

    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options

    chrome_options = Options()

    chrome_options.add_argument("--start-maximized")

    #anti-detection settings
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(options=chrome_options)

    #remove selenium flag
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )

    return driver

In [13]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def fetch_yelp_reviews(url):

    driver = create_driver()
    reviews = []

    driver.get(url)

    # ✅ FIX 1 — WAIT FOR PAGE TO LOAD
    wait = WebDriverWait(driver, 20)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "p")))

    # ✅ FIX 2 — SCROLL TO LOAD REVIEWS
    for _ in range(5):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

    # now extract reviews
    elements = driver.find_elements(By.TAG_NAME, "p")

    for el in elements:
        text = el.text.strip()

        if len(text) > 80:
            reviews.append(text)

    driver.quit()

    return reviews

In [7]:
def analyze_yelp_business(url):

    reviews = fetch_yelp_reviews(url)

    print("Reviews collected:", len(reviews))

    if len(reviews) == 0:
        print("No reviews extracted")
        return

    clean_reviews = [preprocess_text(r) for r in reviews]

    X_text = tfidf.transform(clean_reviews)

    review_length = [[len(r.split())] for r in clean_reviews]

    # placeholder rating (3 stars as neutral value)
    rating_placeholder = [[3] for _ in clean_reviews]

    X_live = sp.hstack([X_text, review_length, rating_placeholder])

    predictions = iso_model.predict(X_live)

    suspicious = [1 if p == -1 else 0 for p in predictions]

    suspicious_percentage = (sum(suspicious) / len(suspicious)) * 100

    print("\nSuspicious review percentage:", round(suspicious_percentage,2), "%")

    if suspicious_percentage > 20:
        print("⚠️ Business may have suspicious review activity")
    else:
        print("✅ Reviews appear mostly genuine")

    for review, pred in zip(reviews, predictions):
        label = "Suspicious" if pred == -1 else "Normal"
        if label == "Suspicious":
            print(f"\n[{label}] {review[:50]}...")

In [16]:
url = "https://www.yelp.com/biz/north-beach-pizza-san-francisco-4"

analyze_yelp_business(url)

Reviews collected: 20

Suspicious review percentage: 0.0 %
✅ Reviews appear mostly genuine
